# Kumo graph visualizer demo

Select a Python kernel with the combined `feat/graph-visualize` branch installed. Run natively on **Apple Silicon with macOS 14 or newer** using a wheel built from this branch, or in a supported Linux environment. A Mac does not need Docker for this demo.

The notebook installs nothing and uses synthetic CSV files, with no credentials, training, or graph snapshots. It creates a unique `kumo-visualizer-demo-*` directory below the notebook working directory. CSV files and exported SVGs remain there after the kernel stops.

In [ ]:
import csv
import sys
import tempfile
from pathlib import Path

from kumo.engine import Edge, FileConnector, Graph, Table

if not hasattr(Graph, "visualize"):
    raise RuntimeError(
        "This kernel does not have Graph.visualize(). Select the environment "
        "with feat/graph-visualize installed and restart the kernel."
    )
print(f"Python: {sys.executable}")

In [ ]:
demo_directory = Path(tempfile.mkdtemp(prefix="kumo-visualizer-demo-", dir=Path.cwd()))
data_directory = demo_directory / "data"
data_directory.mkdir()

tables = {
    "customers": (
        ["customer_id", "age", "loyalty_score"],
        [(1, 28, 80), (2, 42, 95), (3, 35, 60)],
    ),
    "orders": (
        ["order_id", "customer_id", "total"],
        [(101, 1, 45.0), (102, 2, 30.0), (103, 1, 20.0), (104, 3, 15.0)],
    ),
    "order_items": (
        ["item_id", "order_id", "product_id", "quantity"],
        [(1001, 101, 10, 2), (1002, 101, 11, 1), (1003, 102, 12, 1),
         (1004, 103, 10, 1), (1005, 104, 11, 3)],
    ),
    "products": (
        ["product_id", "price", "stock"],
        [(10, 20.0, 100), (11, 5.0, 250), (12, 30.0, 75)],
    ),
}
for name, (columns, rows) in tables.items():
    path = data_directory / f"{name}.csv"
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.writer(stream)
        writer.writerow(columns)
        writer.writerows(rows)
    print(path)

## Connect the tables

Declare primary keys and foreign-key relationships. `Table.from_source_table()` reads the small local CSV schemas; visualization then uses this resolved metadata without materializing a graph.

In [ ]:
files = FileConnector(root_dir=str(data_directory))
customers = Table.from_source_table(files["customers"], primary_key="customer_id")
orders = Table.from_source_table(files["orders"], primary_key="order_id")
order_items = Table.from_source_table(files["order_items"], primary_key="item_id")
products = Table.from_source_table(files["products"], primary_key="product_id")

graph = Graph(
    tables={
        "Customers": customers,
        "Orders": orders,
        "OrderItems": order_items,
        "Products": products,
    },
    edges=[
        Edge("Orders", "customer_id", "Customers"),
        Edge("OrderItems", "order_id", "Orders"),
        Edge("OrderItems", "product_id", "Products"),
    ],
)

## Show relationships

The default view displays primary and foreign keys. The next cell adds non-key columns, bounded to 10 rows per table for graphs of at most 10 tables and 5 rows for larger graphs. Key rows are always retained.

In [ ]:
graph.visualize()

In [ ]:
graph.visualize(show_cols=True)

In [ ]:
keys_svg = demo_directory / "relationships.svg"
detail_svg = demo_directory / "relationships-with-columns.svg"
graph.visualize(keys_svg)
graph.visualize(detail_svg, show_cols=True)
print(f"Keys-only SVG: {keys_svg}")
print(f"Detailed SVG:  {detail_svg}")

## Try your own graph

Replace the four synthetic `Table` objects with your resolved tables and update the aliases and `Edge` definitions. If you already have a `Graph`, call `your_graph.visualize()` directly, or `your_graph.visualize("my-graph.svg", show_cols=True)` to save a detailed view. No `snapshot()` call is needed.